# Notebook 1 - Datasheet Ingestion (PDF -> chunks -> Qdrant)

Flow:

1. **Configuration** - paste the PDF path
2. **Extract** text, headings, tables and figure captions from the PDF (pdfplumber)
3. **Chunk** - structure-aware chunks (section text, one chunk per spec-table parameter, row groups for other tables, captions)
4. **Embed + store** - dense vectors (`BAAI/bge-small-en-v1.5`) and sparse keyword vectors (BM25-style) in a **Qdrant** collection

Run the cells top to bottom. Re-running for the same PDF replaces that PDF's points (no duplicates).
Retrieval lives in `02_retrieval.ipynb`.

## 1. Configuration - paste the PDF path here

In [1]:
from pathlib import Path

# ==== Paste your PDF path here ====
PDF_PATH = r"data/pdfs/ltc3310s-3310s-1-3310-3310-1.pdf"

# ---- Qdrant ----
QDRANT_MODE = "server"            # "server": Qdrant running at QDRANT_URL (Docker / Rancher Desktop)
                                  # "local" : embedded Qdrant in data/qdrant_local (no server, single process only)
QDRANT_URL = "http://localhost:6333"
QDRANT_API_KEY = None             # set if your Qdrant server requires one
COLLECTION = "semiconductor_datasheets"
QDRANT_WAIT_TIMEOUT_S = 900       # how long step 4 waits for the Qdrant server to come up

# ---- Embeddings ----
EMBED_MODEL = "BAAI/bge-small-en-v1.5"   # 384-dim, 512-token input, runs on CPU
EMBED_BATCH_SIZE = 32

# ---- Chunking ----
TEXT_CHUNK_TARGET_TOKENS = 350    # text chunks are packed up to about this size
CHUNK_MAX_TOKENS = 480            # hard cap: bge-small silently truncates beyond 512 tokens
HEADER_MARGIN = 0.05              # top 5% of each page is treated as running header
FOOTER_MARGIN = 0.07              # bottom 7% is treated as footer

PROJECT_DIR = Path.cwd()          # VS Code runs the notebook from its own folder

## 2. Imports and setup

In [2]:
import hashlib
import json
import os
import re
import statistics
import time
import uuid
import zlib
from collections import Counter
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone

import pandas as pd
import pdfplumber

# Corporate proxies re-sign HTTPS: trust the Windows certificate store for model downloads.
try:
    import truststore
    truststore.inject_into_ssl()
except ImportError:
    pass
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")  # the Xet downloader ignores the system certificate store

pd.set_option("display.max_colwidth", 120)

pdf_path = Path(PDF_PATH.strip().strip('"')).expanduser()
assert pdf_path.is_file(), f"PDF not found: {pdf_path}"
assert pdf_path.suffix.lower() == ".pdf", f"Not a PDF: {pdf_path}"

DOC_SHA = hashlib.sha256(pdf_path.read_bytes()).hexdigest()
OUT_DIR = PROJECT_DIR / "data" / "processed" / DOC_SHA[:16]
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"PDF      : {pdf_path.name}  ({pdf_path.stat().st_size / 1e6:.1f} MB)")
print(f"SHA-256  : {DOC_SHA}")
print(f"Outputs  : {OUT_DIR}")

PDF      : ltc3310s-3310s-1-3310-3310-1.pdf  (2.8 MB)
SHA-256  : 1b6c68c71c2763f78404ab4cd6fbd0b07d5666441e7b151ddb65a31ffd496ab5
Outputs  : c:\Users\anshika.prasad\workspace\semiconductor_datasheet_RAG_notebook\data\processed\1b6c68c71c2763f7


## 3. Extract text, tables and figures from the PDF

Per page:
- running header / footer bands are cropped off
- **spec tables** without ruling lines (`Parameter | Conditions | Min | Typ | Max | Unit`) are rebuilt from word
  positions, anchored on the Min / Typ / Max / Unit header (step 3a)
- ruled tables are detected with `find_tables`; implausible ones (graph grids, axes) are dropped
- **two-column pages** are read column by column (left, then right), with full-width lines kept whole
- remaining text lines are grouped into paragraphs; **headings** are detected from font size / bold
- `Figure N.` lines become **captions**
- pages with almost no extractable text are flagged as needing OCR (not indexed)

### 3a. Unruled specification tables

Electrical-characteristics tables in ADI / Linear Technology datasheets are aligned text with no lines, so a
ruled-table finder misses them. Columns are anchored on the header words; sub/superscripts are re-attached
(`V` + `IN` -> `VIN`); parameter hierarchy (GROUP > parameter > sub-row) is flattened into full row names.

In [3]:
SUPER = str.maketrans("0123456789,", "⁰¹²³⁴⁵⁶⁷⁸⁹˒")
SPEC_HEADER = ["Parameter", "Conditions", "Min", "Typ", "Max", "Unit"]
FULL_RANGE = "over full operating temperature range"
FOOTNOTES_START = re.compile(r"^(?:Notes?\s*\d|\d{1,2}\s+[A-Z][a-z]+\s)")


@dataclass
class Word:
    x0: float
    x1: float
    top: float
    size: float
    text: str
    bold: bool = False
    marker: bool = False      # Dingbats bullet = "applies over the full temperature range"
    merged: bool = False


def prepare_words(raw):
    """Attach sub/superscripts to the word they follow: 'V' + 'IN' -> 'VIN', 'Error' + '4' -> 'Error⁴'."""
    words = [Word(w["x0"], w["x1"], w["top"], w["size"], w["text"], "bold" in w["fontname"].lower(),
                  "dingbat" in w["fontname"].lower()) for w in raw]
    by_x1 = {}
    for w in words:
        if not w.marker:
            by_x1.setdefault(round(w.x1), []).append(w)
    attached = set()
    for s in sorted(words, key=lambda w: (w.size, w.top, w.x0)):
        if s.marker:
            continue
        near = [m for k in range(round(s.x0) - 2, round(s.x0) + 3) for m in by_x1.get(k, [])]
        host = next((m for m in near if m is not s and id(m) not in attached and m.size >= s.size * 1.15
                     and -1.5 <= s.x0 - m.x1 < 1.5 and 0.8 <= abs(s.top - m.top) < 6), None)
        if host is None:
            continue
        sup = s.top < host.top + 1.5
        host.text += s.text.translate(SUPER) if sup and re.fullmatch(r"[\d,]+", s.text) else s.text
        host.x1, host.merged = s.x1, True
        attached.add(id(s))
    glued = []
    for w in sorted((w for w in words if id(w) not in attached), key=lambda w: (round(w.top), w.x0)):
        prev = glued[-1] if glued else None
        if prev and prev.merged and abs(w.top - prev.top) < 3 and 0 <= w.x0 - prev.x1 < 0.8 and not w.marker:
            prev.text += w.text            # "(V" "IN" ")" -> "(VIN)"
            prev.x1 = w.x1
        else:
            glued.append(w)
    return glued


def group_lines(words):
    lines = []
    for w in sorted(words, key=lambda w: (w.top, w.x0)):
        for ln in lines:
            if abs(ln[0].top - w.top) <= 3:
                ln.append(w)
                break
        else:
            lines.append([w])
    for ln in lines:
        ln.sort(key=lambda w: w.x0)
    return sorted(lines, key=lambda ln: min(w.top for w in ln))


def find_spec_header(line):
    names = {w.text.strip(":").lower(): w for w in line}
    mn, ty, mx = names.get("min"), names.get("typ"), names.get("max")
    un = names.get("unit") or names.get("units")
    if not (mn and ty and mx and un):
        return None
    cond = next((w for w in line if w.text.lower().startswith(("condition", "test"))), None)
    return {"param_x0": line[0].x0, "cond_x0": cond.x0 if cond else line[0].x1 + 40,
            "min": (mn.x0 + mn.x1) / 2, "typ": (ty.x0 + ty.x1) / 2, "max": (mx.x0 + mx.x1) / 2, "unit_x0": un.x0}


def spec_column(w, h):
    cx = (w.x0 + w.x1) / 2
    if w.x0 < h["cond_x0"] - 3:
        return "param"
    if cx < h["min"] - (h["typ"] - h["min"]) / 2:
        return "cond"
    if cx < (h["min"] + h["typ"]) / 2:
        return "min"
    if cx < (h["typ"] + h["max"]) / 2:
        return "typ"
    return "max" if w.x0 < h["unit_x0"] - 4 else "unit"


def spec_cells(line, h):
    cells = {k: [] for k in ("param", "cond", "min", "typ", "max", "unit")}
    full = False
    for w in line:
        if w.marker:
            full = True
        else:
            cells[spec_column(w, h)].append(w)
    out = {k: " ".join(w.text for w in v).strip() for k, v in cells.items()}
    out["text"] = " ".join(w.text for w in line if not w.marker)
    out["param_x0"] = cells["param"][0].x0 if cells["param"] else None
    out["full"] = full
    out["has_values"] = any(out[k] for k in ("min", "typ", "max", "unit"))
    return out


def join_unique(*parts):
    out = []
    for p in parts:
        if p and p not in out:
            out.append(p)
    return " | ".join(out)


def is_upper(s):
    letters = [ch for ch in s if ch.isalpha()]
    return len(letters) >= 3 and all(ch.isupper() for ch in letters)


def build_spec_rows(lines, h):
    """GROUP (all caps) > parameter > optional indented sub-label > indented child row with values.
    Children whose text is a condition ('OSR = 1') extend the parent's conditions instead of its name."""
    cells = [spec_cells(ln, h) for ln in lines]
    xs = [c["param_x0"] for c in cells if c["param_x0"] is not None and not is_upper(c["param"])]
    base = min(xs) if xs else h["param_x0"]
    rows, group_cond, top, sub, fresh = [], "", None, None, False

    def parent_cond():
        out = [group_cond]
        if top and not top["has_values"]:
            out.append(top["cond"])
        if sub:
            out.append(sub["cond"])
        return out

    for c in cells:
        param, cond = c["param"], c["cond"]
        if FOOTNOTES_START.match(c["text"]):
            break
        indented = c["param_x0"] is not None and c["param_x0"] > base + 3
        if param and not indented and not c["has_values"] and is_upper(param):
            rows.append([param, "", "", "", "", ""])          # group label row
            group_cond, top, sub, fresh = cond, None, None, False
            continue
        if not c["has_values"]:
            if param and not indented:
                top, sub, fresh = {"name": param, "cond": cond, "has_values": False}, None, True
            elif param:
                if sub and c["param_x0"] <= sub["x0"] + 3:
                    sub = None
                sub, fresh = {"name": param, "cond": cond, "x0": c["param_x0"]}, True
            elif cond:
                if fresh and (sub or top):
                    tgt = sub or top
                    tgt["cond"] = f"{tgt['cond']} {cond}".strip()
                elif rows:
                    rows[-1][1] = f"{rows[-1][1]} {cond}".strip()
            continue
        if param and not indented:
            top, sub = {"name": param, "cond": cond, "has_values": True}, None
            name, conds = param, [group_cond, cond]
        else:
            if param and sub and c["param_x0"] <= sub["x0"] + 3:
                sub = None
            as_cond = "=" in param
            name = " ".join(p for p in [top["name"] if top else "", sub["name"] if sub else "",
                                        "" if as_cond else param] if p)
            conds = parent_cond() + [param if as_cond else "", cond]
        rows.append([name, join_unique(*conds, FULL_RANGE if c["full"] else ""), c["min"], c["typ"], c["max"], c["unit"]])
        fresh = False
    return rows


def line_text(ln):
    return " ".join("●" if w.marker else w.text for w in ln)


def default_conditions(above):
    """The '... unless otherwise noted.' paragraph above a spec table: conditions for every row."""
    tail = above[-8:]
    closes = lambda j: re.search(r"otherwise\s+(noted|specified|stated)",
                                 (line_text(tail[j - 1]) + " " if j > 0 else "") + line_text(tail[j]), re.I)
    end = next((j for j in range(len(tail) - 1, -1, -1) if closes(j)), None)
    if end is None:
        return ""
    start = end
    while start > 0:
        prev, cur = tail[start - 1], tail[start]
        gap = min(w.top for w in cur) - min(w.top for w in prev)
        if gap > 2.2 * max(w.size for w in prev) or re.match(r"^Table\s+\d+", line_text(prev)) or is_upper(line_text(prev)):
            break
        start -= 1
    return re.sub(r"\s+", " ", " ".join(line_text(ln) for ln in tail[start:end + 1])).strip()


def extract_spec_tables(area):
    raw = area.extract_words(extra_attrs=["fontname", "size"], keep_blank_chars=False)
    if not raw:
        return []
    lines = group_lines(prepare_words(raw))
    heads = [(i, h) for i, h in ((i, find_spec_header(ln)) for i, ln in enumerate(lines)) if h]
    out = []
    for k, (i, h) in enumerate(heads):
        body = lines[i + 1: heads[k + 1][0] if k + 1 < len(heads) else len(lines)]
        rows = build_spec_rows(body, h)
        if not rows:
            continue
        title, top = None, min(w.top for w in lines[i])
        if i > 0 and re.match(r"^Table\s+\d+", line_text(lines[i - 1])) and top - min(w.top for w in lines[i - 1]) < 30:
            title, top = line_text(lines[i - 1]), min(w.top for w in lines[i - 1])
        floor = heads[k - 1][0] + 1 if k > 0 else 0
        bottom = max((w.top + w.size for ln in body for w in ln), default=top)
        px0, ptop, px1, pbottom = (float(v) for v in area.bbox)
        out.append({"bbox": (px0, max(top - 1, ptop), px1, min(bottom + 2, pbottom)),
                    "rows": [SPEC_HEADER] + rows, "title": title, "kind": "spec",
                    "default_conditions": default_conditions(lines[floor:i - (1 if title else 0)])})
    return out

### 3b. Page layout, ruled tables, headings, paragraphs

In [4]:
@dataclass
class Block:
    kind: str                 # heading | text | table | caption
    page: int
    text: str = ""
    level: int = 0            # heading level (1 = top)
    top: float = 0.0          # y position
    table_index: int | None = None
    order: tuple = ()         # reading-order key: (band, column, top)


@dataclass
class PageData:
    page: int
    width: float
    height: float
    blocks: list = field(default_factory=list)
    tables: list = field(default_factory=list)     # list of {"title", "rows", "bbox", "kind", "default_conditions"}
    columns: int = 1
    char_count: int = 0
    image_count: int = 0
    needs_ocr: bool = False


TABLE_TITLE_RE = re.compile(r"^Table\s+\d+[.:]?\s*", re.I)
CAPTION_RE = re.compile(r"^(Figure|Fig\.)\s*\d+[a-z]?[.:]?\s+\S", re.I)


def clean_cell(cell):
    return re.sub(r"\s+", " ", cell or "").strip()


def plausible_table(rows):
    """Drop false positives (graph axes, gridlines): need >= 2 rows, >= 2 columns, mostly filled cells."""
    cells = [c for r in rows for c in r]
    return (len(rows) >= 2 and max((len(r) for r in rows), default=0) >= 2 and bool(cells)
            and sum(bool(c) for c in cells) / len(cells) >= 0.4)


def overlaps(a, b):
    return a[0] < b[2] and b[0] < a[2] and a[1] < b[3] and b[1] < a[3]


def body_font_size(lines):
    weights = Counter(round(c["size"], 1) for ln in lines for c in ln["chars"] if c["text"].strip())
    return weights.most_common(1)[0][0] if weights else 10.0


def line_style(line):
    chars = [c for c in line["chars"] if c["text"].strip()]
    if not chars:
        return 0.0, False
    size = statistics.median(c["size"] for c in chars)
    bold = sum("bold" in c["fontname"].lower() for c in chars) / len(chars) > 0.6
    return size, bold


def heading_level(text, size, bold, body):
    if len(text) > 90 or len(text) < 3 or text.endswith((".", ",", ";")):
        return 0
    if text[0].islower() or sum(ch.isalpha() for ch in text) < 3:   # wrapped sentence, "( )", axis ticks
        return 0
    if size < body * 0.95:
        return 0
    if size >= body * 1.3:
        return 1
    if size >= body * 1.15:
        return 1 if text.isupper() else 2
    numbered = re.match(r"^\d+(\.\d+)*\s+\S", text) is not None
    if bold and (text.isupper() or numbered):
        return 1 if text.isupper() else 2
    if bold and len(text.split()) <= 8:
        return 2
    return 0


def join_lines(lines):
    out = lines[0]
    for ln in lines[1:]:
        out = out[:-1] + ln if out.endswith("-") and ln[:1].islower() else out + " " + ln
    return out


def lines_to_blocks(lines, skip, body, page_no):
    """lines: [(order_key, line)] already in reading order. A new column / band always starts a new paragraph."""
    blocks, cur, cur_order, prev_bottom, prev_region = [], [], (), None, None
    line_h = body * 1.25

    def flush():
        nonlocal cur
        if cur:
            blocks.append(Block("text", page_no, text=join_lines(cur), top=cur_order[2], order=cur_order))
            cur = []

    for i, (key, ln) in enumerate(lines):
        text = ln["text"].strip()
        if i in skip or not text:
            continue
        if key[:2] != prev_region:
            flush()
            prev_region, prev_bottom = key[:2], None
        if CAPTION_RE.match(text):
            flush()
            blocks.append(Block("caption", page_no, text=text, top=ln["top"], order=key))
            prev_bottom = ln["bottom"]
            continue
        size, bold = line_style(ln)
        level = heading_level(text, size, bold, body)
        if level:
            flush()
            blocks.append(Block("heading", page_no, text=text, level=level, top=ln["top"], order=key))
            prev_bottom = ln["bottom"]
            continue
        if cur and prev_bottom is not None and ln["top"] - prev_bottom > line_h * 0.9:
            flush()   # vertical gap = new paragraph
        if not cur:
            cur_order = key
        cur.append(text)
        prev_bottom = ln["bottom"]
    flush()
    return blocks


def find_gutter(words, bbox):
    """x of the white gap between two text columns, or None for a single-column page.
    Words crossing the gap (full-width headings) are returned so their lines can be kept whole."""
    x0, _, x1, _ = bbox
    if len(words) < 40:
        return None
    crossing = {x: sum(w["x0"] < x < w["x1"] for w in words)
                for x in range(int(x0 + (x1 - x0) * 0.35), int(x0 + (x1 - x0) * 0.65))}
    if not crossing:
        return None
    low = min(crossing.values())
    candidates = [x for x, n in crossing.items() if n == low]
    g = candidates[len(candidates) // 2]
    left = sum(w["x1"] <= g for w in words)
    right = sum(w["x0"] >= g for w in words)
    if low <= max(2, 0.03 * len(words)) and min(left, right) >= 0.2 * len(words):
        return g, [(w["top"], w["bottom"]) for w in words if w["x0"] < g < w["x1"]]
    return None


def merge_intervals(spans, pad=1.0):
    out = []
    for t, b in sorted(spans):
        if out and t - pad <= out[-1][1]:
            out[-1][1] = max(out[-1][1], b)
        else:
            out.append([t, b])
    return out


class Layout:
    """Reading order: bands separated by full-width lines; inside a band, left column then right column."""

    def __init__(self, gutter=None, bands=()):
        self.g, self.bands = gutter, list(bands)

    def key(self, top, x0, x1):
        if self.g is None:
            return (0, 0, top)
        for k, (bt, bb) in enumerate(self.bands):
            if bt - 1 <= top <= bb + 1:
                return (2 * k + 1, 0, top)
        seg = sum(bb < top for _, bb in self.bands)
        if x0 < self.g < x1:                     # full-width object (e.g. a wide table)
            return (2 * seg, 0, top)
        return (2 * seg, 0 if (x0 + x1) / 2 < self.g else 1, top)


def ordered_lines(area):
    words = area.extract_words()
    gut = find_gutter(words, area.bbox)
    kw = dict(return_chars=True, strip=True)
    if not gut:
        layout = Layout()
        return [(layout.key(ln["top"], ln["x0"], ln["x1"]), ln) for ln in area.extract_text_lines(**kw) or []], layout
    g, crossing = gut
    bands = merge_intervals(crossing)
    layout = Layout(g, bands)
    is_char = lambda o: o.get("object_type") == "char"
    mid_y = lambda o: (o["top"] + o["bottom"]) / 2
    in_band = lambda o: any(t - 1 <= mid_y(o) <= b + 1 for t, b in bands)
    left = lambda o: (o["x0"] + o["x1"]) / 2 < g
    streams = [
        area.filter(lambda o: not is_char(o) or in_band(o)),
        area.filter(lambda o: not is_char(o) or (not in_band(o) and left(o))),
        area.filter(lambda o: not is_char(o) or (not in_band(o) and not left(o))),
    ]
    lines = [(layout.key(ln["top"], ln["x0"], ln["x1"]), ln) for s in streams for ln in s.extract_text_lines(**kw) or []]
    lines.sort(key=lambda kl: kl[0])
    return lines, layout


def parse_page(page, page_no):
    w, h = float(page.width), float(page.height)
    out = PageData(page=page_no, width=w, height=h, image_count=len(page.images))
    body = page.crop((0, h * HEADER_MARGIN, w, h * (1 - FOOTER_MARGIN)))

    # 1. unruled spec tables, then ruled tables that do not overlap them
    try:
        tables = extract_spec_tables(body)
    except Exception:          # a malformed page must not stop the whole document
        tables = []
    try:
        found = body.find_tables()
    except Exception:
        found = []
    for t in found:
        rows = [[clean_cell(c) for c in r] for r in t.extract()]
        rows = [r for r in rows if any(r)]
        if plausible_table(rows) and not any(overlaps(t.bbox, o["bbox"]) for o in tables):
            tables.append({"bbox": tuple(t.bbox), "rows": rows, "title": None, "kind": "ruled", "default_conditions": ""})

    # 2. text outside the tables, in column-aware reading order
    text_area = body
    for t in tables:
        text_area = text_area.outside_bbox(t["bbox"])
    lines, layout = ordered_lines(text_area)
    out.columns = 1 if layout.g is None else 2
    body_size = body_font_size([ln for _, ln in lines])

    used = set()
    for t in tables:
        x0, top, x1, _ = t["bbox"]
        if not t["title"]:
            for i, (_, ln) in enumerate(lines):   # a "Table 3. ..." line just above the table is its title
                if 0 <= top - ln["bottom"] <= 45 and TABLE_TITLE_RE.match(ln["text"]):
                    t["title"] = ln["text"].strip()
                    used.add(i)
        out.tables.append(t)
    blocks = [Block("table", page_no, top=t["bbox"][1], table_index=i, order=layout.key(t["bbox"][1], t["bbox"][0], t["bbox"][2]))
              for i, t in enumerate(out.tables)]
    blocks += lines_to_blocks(lines, used, body_size, page_no)
    out.blocks = sorted(blocks, key=lambda b: b.order)

    out.char_count = sum(len(ln["text"]) for _, ln in lines) + sum(len(c) for t in tables for r in t["rows"] for c in r)
    out.needs_ocr = out.char_count < 30
    return out


def strip_repeated_boilerplate(pages):
    """Short lines repeated on most pages (part-number running heads, copyright, 'Rev. B | Page 7 of 40') are noise."""
    if len(pages) < 4:
        return 0
    norm = lambda t: re.sub(r"\d+", "#", t.lower()).strip()
    short = lambda b: b.kind in ("text", "heading") and len(b.text) < 80
    seen = Counter(k for p in pages for k in {norm(b.text) for b in p.blocks if short(b)})
    repeated = {k for k, n in seen.items() if n >= max(3, len(pages) * 0.5)}
    removed = 0
    for p in pages:
        keep = [b for b in p.blocks if not (short(b) and norm(b.text) in repeated)]
        removed += len(p.blocks) - len(keep)
        p.blocks = keep
    return removed

In [5]:
t0 = time.time()
with pdfplumber.open(pdf_path) as pdf:
    pdf_info = pdf.metadata or {}
    pages = [parse_page(pg, i) for i, pg in enumerate(pdf.pages, 1)]
removed = strip_repeated_boilerplate(pages)
print(f"Parsed {len(pages)} pages in {time.time() - t0:.1f}s; removed {removed} repeated header/footer lines")

page_summary = pd.DataFrame([{
    "page": p.page,
    "columns": p.columns,
    "headings": sum(b.kind == "heading" for b in p.blocks),
    "paragraphs": sum(b.kind == "text" for b in p.blocks),
    "tables": len(p.tables),
    "captions": sum(b.kind == "caption" for b in p.blocks),
    "images": p.image_count,
    "chars": p.char_count,
    "needs_ocr": p.needs_ocr,
} for p in pages])
page_summary

Parsed 32 pages in 14.2s; removed 92 repeated header/footer lines


,page,columns,headings,paragraphs,tables,captions,images,chars,needs_ocr
0,1,2,7,7,0,0,0,2455,False
1,2,2,3,6,2,0,0,1959,False
2,3,2,5,9,3,0,0,2654,False
3,4,2,1,2,1,0,0,2124,False
4,5,2,21,14,4,0,0,1773,False
5,6,2,10,20,1,0,0,1203,False
6,7,2,9,25,0,0,0,1150,False
7,8,2,12,22,0,0,0,844,False
8,9,2,8,16,0,0,3,1073,False
9,10,2,1,2,0,0,0,3846,False


### Document metadata (title, part numbers, manufacturer, revision)

In [6]:
PART_RE = re.compile(r"\b(?:[A-Z]{2,5}\d{3,5}[A-Z0-9]*(?:-\d{1,2})?)\b")
MANUFACTURERS = ["Analog Devices", "Linear Technology", "Texas Instruments", "Maxim Integrated", "Microchip",
                 "STMicroelectronics", "NXP", "Infineon", "onsemi", "ON Semiconductor", "Renesas", "Nexperia",
                 "ROHM", "Vishay", "Diodes Incorporated", "Monolithic Power Systems"]

first_pages_text = " ".join(b.text for p in pages[:2] for b in p.blocks if b.kind in ("heading", "text"))
all_text = " ".join(b.text for p in pages for b in p.blocks if b.kind in ("heading", "text"))

headings_p1 = [b.text for b in pages[0].blocks if b.kind == "heading"]
title = (pdf_info.get("Title") or "").strip() or (headings_p1[0] if headings_p1 else pdf_path.stem)

# Parts: those named in the title (e.g. "LTC3310S/LTC3310S-1/LTC3310/LTC3310-1"); otherwise the part numbers
# repeated on the first two pages (this can pick up ordering codes - edit doc_meta below if so)
parts = list(dict.fromkeys(PART_RE.findall(title)))
if not parts:
    part_counts = Counter(PART_RE.findall(first_pages_text))
    parts = [p for p, n in part_counts.most_common() if n >= 2][:12] or [p for p, _ in part_counts.most_common(3)]
manufacturer = next((m for m in MANUFACTURERS if m.lower() in all_text.lower()), "")
rev = re.search(r"\bRev(?:ision)?\.?\s*([A-Z0-9]{1,3})\b", all_text)

doc_meta = {
    "doc_sha": DOC_SHA,
    "file_name": pdf_path.name,
    "source_path": str(pdf_path),
    "title": title,
    "parts": parts,
    "manufacturer": manufacturer,
    "revision": rev.group(1) if rev else "",
    "page_count": len(pages),
}
doc_meta   # edit any field by hand here if the heuristics got it wrong, e.g. doc_meta["parts"] = ["LTC3310S"]

{'doc_sha': '1b6c68c71c2763f78404ab4cd6fbd0b07d5666441e7b151ddb65a31ffd496ab5',
 'file_name': 'ltc3310s-3310s-1-3310-3310-1.pdf',
 'source_path': 'C:\\Users\\anshika.prasad\\workspace\\semiconductor_datasheet_RAG\\data\\inbox\\ltc3310s-3310s-1-3310-3310-1.pdf',
 'title': 'LTC3310S/LTC3310S-1/LTC3310/LTC3310-1 (Rev. F)',
 'parts': ['LTC3310S', 'LTC3310S-1', 'LTC3310', 'LTC3310-1'],
 'manufacturer': 'Analog Devices',
 'revision': 'F',
 'page_count': 32}

### Preview: headings and the first table

In [7]:
outline = [("  " * (b.level - 1)) + f"p{b.page}: {b.text}" for p in pages for b in p.blocks if b.kind == "heading"]
print("\n".join(outline[:40]), "\n..." if len(outline) > 40 else "")

first_table = next((t for p in pages for t in p.tables if t["kind"] == "spec"), None) or next((t for p in pages for t in p.tables), None)
if first_table:
    print(f"\nFirst {first_table['kind']} table:", first_table["title"] or "(untitled)", "|", first_table.get("default_conditions", "")[:200])
    display(pd.DataFrame(first_table["rows"][1:], columns=[c or f"col{i}" for i, c in enumerate(first_table["rows"][0])]).head(15))

p1: 5V, 10A Synchronous Step-Down
p1: Silent Switcher 2 in 3mm × 3mm LQFN
p1: FEATURES
  p1: Emissions (LTC3310S and LTC3310S-1)
p1: APPLICATIONS
p1: TYPICAL APPLICATION
p1: DESCRIPTION
p2: ABSOLUTE MAXIMUM RATINGS
p2: ORDER INFORMATION
p2: PIN CONFIGURATION
p3: ORDER INFORMATION
p3: LEAD FREE FINISH AUTOMOTIVE PRODUCTS** PART MARKING* PACKAGE DESCRIPTION TEMPERATURE RANGE
p3: LTC3310
p3: LTC3310-1
p3: ELECTRICAL CHARACTERISTICS
p4: ELECTRICAL CHARACTERISTICS
p5: TYPICAL PERFORMANCE CHARACTERISTICS
p5: A IN
p5: OUT
p5: OUT
p5: OUT
p5: V Line Regulation
p5: OUT
p5: (LTC3310S-1, LTC3310-1)
p5: Vout Line Regulation
p5: V Load Regulation OUT
p5: V Line Regulation, OUT
p5: (LTC3310S-1, LTC3310-1)
p5: V = 1.2V OUT
p5: Vout Load Regulation
p5: OUT
p5: Efficiency, Forced Continuous
p5: PMOS Current Limit Mode, V = 3.3V IN
p5: Forced Continuous Mode Operation
p5: V IN = 3.3V, V OUT = 1.2V V IN = 3.3V
p5: Pulse Skip Mode
p5: Pulse Skip Mode Operation
p6: TYPICAL PERFORMANCE CHARACTERISTICS
p6: A

,Parameter,Conditions,Min,Typ,Max,Unit
0,Operating Supply Voltage (VIN ),over full operating temperature range,2.25,,5.5,V
1,VIN Undervoltage Lockout,VIN Rising | over full operating temperature range,2.0,2.1,2.2,V
2,VIN Undervoltage Lockout Hysteresis,,,150,,mV
3,VIN Quiescent Current,(Note 4),,1.3,2.0,mA
4,VIN Quiescent Current in Shutdown,VEN = 0.1V,,1,2,μA
5,EN Threshold,VEN Rising | over full operating temperature range,0.375,0.4,0.425,V
6,EN Hysteresis,,,60,,mV
7,EN Pin Leakage Current,VEN = 0.4V,,,±20,nA
8,Regulated Feedback Voltage (VFB ),over full operating temperature range,495,500,505,mV
9,Regulated Output Voltage,"LTC3310S-1, LTC3310-1, IOUT = 4A | over full operating temperature range",0.990,1.000,1.010,V


In [8]:
# Save the extraction so it can be inspected without re-parsing
(OUT_DIR / "parsed.json").write_text(json.dumps(
    {"meta": doc_meta, "pages": [asdict(p) for p in pages]}, indent=1, ensure_ascii=False), encoding="utf-8")
print("Saved", OUT_DIR / "parsed.json")

Saved c:\Users\anshika.prasad\workspace\semiconductor_datasheet_RAG_notebook\data\processed\1b6c68c71c2763f7\parsed.json


## 4. Chunk the document

Rules (carried over from the bake-off in the earlier project, where chunking moved recall far more than the model):

- every chunk starts with its **section path**, e.g. `[ELECTRICAL CHARACTERISTICS > Switching Regulator]`
- **text**: paragraphs packed to ~350 tokens, never crossing a heading or page; never over the model's input limit
- **spec tables** (header has MIN / TYP / MAX): **one chunk per parameter**, with the column header repeated
- **other tables**: row groups that fit the limit, header repeated in each
- **captions**: one chunk per figure caption
- schematic / graph label text ("VIN 2.25V TO 5.5V 22uF EN PGOOD") is **not indexed**; it matches everything

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)

def count_tokens(text):
    return len(tokenizer(text, add_special_tokens=True, verbose=False)["input_ids"])

print(f"Tokenizer for {EMBED_MODEL} loaded; model max length = {tokenizer.model_max_length}")

Tokenizer for BAAI/bge-small-en-v1.5 loaded; model max length = 512


In [10]:
@dataclass
class Chunk:
    chunk_id: str
    type: str                 # text | table | caption
    text: str
    section_path: str
    page: int
    parts: list
    table_id: str = ""
    table_conditions: str = ""   # spec tables: "... unless otherwise noted" conditions for every row
    tokens: int = 0


PROSE_WORD = re.compile(r"^[(\"']?[a-z][a-z'\-]{2,}[.,;:)\"']*$")
KEEP_LABEL_SECTIONS = re.compile(r"ORDER|PIN", re.I)    # ordering codes and pin lists are terse but wanted
VALUE_HEADER_RE = re.compile(r"^(MIN|TYP|MAX)\b", re.I)


def is_label_soup(text):
    """Diagram / graph label text has almost no lowercase prose words."""
    tokens = text.split()
    if not tokens:
        return True
    prose = sum(bool(PROSE_WORD.match(t)) for t in tokens)
    return prose / len(tokens) < 0.25 if len(tokens) >= 5 else prose == 0


def split_sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+(?=[A-Z(])", text) if s]


def md_row(row):
    return "| " + " | ".join(c.replace("|", "\\|") for c in row) + " |"


def table_markdown(header, rows):
    width = max([len(header)] + [len(r) for r in rows])
    pad = lambda r: list(r) + [""] * (width - len(r))
    lines = [md_row(pad(header)), "|" + "---|" * width]
    return "\n".join(lines + [md_row(pad(r)) for r in rows])


def parts_in(text):
    """Which of the document's parts a row names; none named -> applies to all."""
    named = [p for p in doc_meta["parts"] if re.search(rf"\b{re.escape(p)}\b", text)]
    return named or doc_meta["parts"]


class Chunker:
    def __init__(self, meta, target=TEXT_CHUNK_TARGET_TOKENS, max_tokens=CHUNK_MAX_TOKENS):
        self.meta, self.target, self.max = meta, target, max_tokens
        self.seq = 0
        self.dropped_labels = []

    def make(self, kind, body, section, page, parts=None, table_id="", table_conditions=""):
        self.seq += 1
        text = f"[{section}]\n{body}" if section else body
        return Chunk(chunk_id=f"{self.meta['doc_sha'][:12]}:{kind}:{page}:{self.seq}", type=kind, text=text,
                     section_path=section, page=page, parts=parts or self.meta["parts"], table_id=table_id,
                     table_conditions=table_conditions, tokens=count_tokens(text))

    # ---- text ----
    def hard_split(self, unit):
        n = count_tokens(unit)
        if n <= self.max:
            return [unit]
        words = unit.split()
        size = max(1, int(len(words) * self.max * 0.85 / n))
        return [" ".join(words[i:i + size]) for i in range(0, len(words), size)]

    def text_chunks(self, paras, section, page):
        if not KEEP_LABEL_SECTIONS.search(section):
            self.dropped_labels += [p for p in paras if is_label_soup(p)]
            paras = [p for p in paras if not is_label_soup(p)]
        units = []
        for p in paras:
            units += split_sentences(p) if count_tokens(p) > self.target else [p]
        units = [piece for u in units for piece in self.hard_split(u)]
        out, cur, cur_tok = [], [], 0
        budget = self.target - count_tokens(f"[{section}]\n")
        for u in units:
            n = count_tokens(u)
            if cur and cur_tok + n > budget:
                out.append(self.make("text", "\n\n".join(cur), section, page))
                cur, cur_tok = [], 0
            cur.append(u)
            cur_tok += n
        if cur:
            out.append(self.make("text", "\n\n".join(cur), section, page))
        return out

    # ---- tables ----
    def split_rows(self, header, rows, head, section):
        budget = self.max - count_tokens(f"[{section}]\n{head}\n{table_markdown(header, [])}")
        groups, cur, used = [], [], 0
        for r in rows:
            n = count_tokens(md_row(r))
            if cur and used + n > budget:
                groups.append(cur)
                cur, used = [], 0
            cur.append(r)
            used += n
        return groups + ([cur] if cur else [])

    def table_chunks(self, table, table_id, page, section):
        rows = table["rows"]
        header, body = rows[0], rows[1:]
        # two-row headers: "PARAMETER | CONDITIONS | ... " then "MIN | TYP | MAX"
        if body and not any(VALUE_HEADER_RE.match(c) for c in header) and sum(bool(VALUE_HEADER_RE.match(c)) for c in body[0]) >= 2:
            header = [f"{a} {b}".strip() for a, b in zip(header, body[0])]
            body = body[1:]
        if not body:
            return []
        title = table["title"] or f"Table on page {page}"
        value_cols = [i for i, c in enumerate(header) if re.search(r"\b(MIN|TYP|MAX)\b", c, re.I)]
        if len(value_cols) >= 2:
            return self.parameter_chunks(header, body, value_cols, title, table_id, page, section,
                                         table.get("default_conditions", ""))
        head = f"{title} (table {table_id}, page {page})"
        groups = self.split_rows(header, body, head + " [rows 99/99]", section)
        out = []
        for i, g in enumerate(groups):
            label = f" [rows {i + 1}/{len(groups)}]" if len(groups) > 1 else ""
            out.append(self.make("table", f"{head}{label}\n{table_markdown(header, g)}", section, page,
                                 parts=parts_in(" ".join(" ".join(r) for r in g)), table_id=table_id))
        return out

    def parameter_chunks(self, header, body, value_cols, title, table_id, page, section, table_conditions=""):
        """One chunk per parameter (its row plus continuation rows). Group-label rows ('DC ACCURACY') are carried along.
        The table-wide conditions paragraph is kept in the payload, not the text: at ~150 words it would dominate
        every small chunk's embedding."""
        groups, label = [], ""
        for row in body:
            filled = [c for c in row if c]
            if len(filled) == 1 and not any(row[i] for i in value_cols if i < len(row)):
                label = filled[0]
                continue
            name = row[0]
            if groups and groups[-1][0] == label and not name:
                groups[-1][2].append(row)
            else:
                groups.append((label, name or (groups[-1][1] if groups else ""), [row]))
        out = []
        for label, name, rows in groups:
            parts = parts_in(" ".join(" ".join(r) for r in rows))
            head = f"{name} - {'/'.join(parts)} - {title} (table {table_id}, page {page})"
            if label:
                head += f"\nGroup: {label}"
            for piece in self.split_rows(header, rows, head, section):
                out.append(self.make("table", f"{head}\n{table_markdown(header, piece)}", section, page,
                                     parts=parts, table_id=table_id, table_conditions=table_conditions))
        return out

    # ---- whole document ----
    def build(self, pages):
        chunks, stack, buf, buf_page = [], [], [], 1
        section = lambda: " > ".join(t for _, t in stack)

        def flush():
            nonlocal buf
            if buf:
                chunks.extend(self.text_chunks(buf, section(), buf_page))
                buf = []

        for p in pages:
            if p.needs_ocr:
                continue
            for b in p.blocks:
                if b.kind == "heading":
                    flush()
                    while stack and stack[-1][0] >= b.level:
                        stack.pop()
                    stack.append((b.level, b.text))
                elif b.kind == "text":
                    if not buf:
                        buf_page = p.page
                    buf.append(b.text)
                elif b.kind == "table":
                    flush()
                    chunks.extend(self.table_chunks(p.tables[b.table_index], f"p{p.page}-t{b.table_index + 1}",
                                                    p.page, section()))
                elif b.kind == "caption":
                    chunks.append(self.make("caption", b.text, section(), p.page))
            flush()   # text never runs across a page boundary
        return chunks

In [11]:
chunker = Chunker(doc_meta)
chunks = chunker.build(pages)

stats = pd.DataFrame([{"type": c.type, "tokens": c.tokens, "page": c.page} for c in chunks])
print(f"{len(chunks)} chunks; {len(chunker.dropped_labels)} diagram-label paragraphs not indexed")
print(f"Longest chunk: {stats.tokens.max()} tokens (cap {CHUNK_MAX_TOKENS}); over cap: {(stats.tokens > CHUNK_MAX_TOKENS).sum()}")
stats.groupby("type").tokens.describe()[["count", "mean", "min", "max"]].round(0)

134 chunks; 271 diagram-label paragraphs not indexed
Longest chunk: 349 tokens (cap 480); over cap: 0


,count,mean,min,max
type,,,,
caption,5.0,20.0,11.0,26.0
table,64.0,119.0,44.0,312.0
text,65.0,147.0,9.0,349.0


In [12]:
# Look at a few chunks of each type
for kind in ("text", "table", "caption"):
    sample = [c for c in chunks if c.type == kind][:2]
    for c in sample:
        print(f"--- {c.chunk_id}  ({c.type}, page {c.page}, {c.tokens} tokens)\n{c.text[:700]}\n")

--- 1b6c68c71c27:text:1:1  (text, page 1, 258 tokens)
[DESCRIPTION]
The LTC®3310S is a very small, low noise, monolithic step-down DC/DC converter capable of providing up to 10A of output current from a 2.25V to 5.5V input supply. The device employs Silent Switcher 2 architecture with internal hot loop bypass capacitors to achieve both low EMI and high efficiency at switching frequencies as high as 5MHz. For systems with higher power requirements, multi-phasing parallel converters is readily implemented. The LTC3310S uses a constant-frequency, peak current mode control architecture for fast transient response. A 500mV reference allows for low voltage outputs. 100% duty cycle operation delivers low drop out. Other features include a power good s

--- 1b6c68c71c27:text:2:2  (text, page 2, 310 tokens)
[PIN CONFIGURATION]
TOP VIEW TUOV ro TTSS HTI TR BF 18 17 16 15 1 14 EN PGOOD 2 13 AGND MODE/SYNC VIN 19 VIN 3 12 PGND 4 11 VIN VIN 5 10 PGND PGND 6 7 8 9 WS WS WS WS LQFN PACKAGE 18-LEAD (3

In [13]:
with open(OUT_DIR / "chunks.jsonl", "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(asdict(c), ensure_ascii=False) + "\n")
print("Saved", OUT_DIR / "chunks.jsonl")

Saved c:\Users\anshika.prasad\workspace\semiconductor_datasheet_RAG_notebook\data\processed\1b6c68c71c2763f7\chunks.jsonl


## 5. Create embeddings and store them in Qdrant

Each point in Qdrant carries two vectors:

| name | what | used for |
|---|---|---|
| `dense` | `bge-small-en-v1.5`, 384-dim, cosine | meaning ("current when the chip is off" ~ "shutdown current") |
| `sparse` | keyword term weights, Qdrant applies **IDF** | exact tokens: part numbers, register names, symbols |

Notebook 2 fuses both (hybrid search).

### 5a. Wait for Qdrant

In `server` mode this cell waits until Qdrant answers at `QDRANT_URL`. Start it with, for example (Rancher Desktop / Docker):

```
docker run -d --name qdrant -p 6333:6333 -p 6334:6334 -v qdrant_storage:/qdrant/storage qdrant/qdrant
```

Set `QDRANT_MODE = "local"` in step 1 to try the notebooks before the server is installed.

In [14]:
from qdrant_client import QdrantClient, models


def connect_qdrant(timeout_s=QDRANT_WAIT_TIMEOUT_S, poll_s=5):
    if QDRANT_MODE == "local":
        path = PROJECT_DIR / "data" / "qdrant_local"
        print(f"Using embedded Qdrant at {path}")
        return QdrantClient(path=str(path))
    deadline, attempt = time.time() + timeout_s, 0
    while True:
        attempt += 1
        try:
            client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=10)
            client.get_collections()
            info = client.info() if hasattr(client, "info") else None
            print(f"Connected to Qdrant at {QDRANT_URL}" + (f" (version {info.version})" if info else ""))
            return client
        except Exception as exc:
            if time.time() > deadline:
                raise RuntimeError(f"Qdrant not reachable at {QDRANT_URL} after {timeout_s}s. "
                                   "Start the Qdrant container, or set QDRANT_MODE = 'local'.") from exc
            if attempt == 1 or attempt % 12 == 0:
                print(f"Waiting for Qdrant at {QDRANT_URL} ... ({type(exc).__name__})")
            time.sleep(poll_s)


client = connect_qdrant()

Connected to Qdrant at http://localhost:6333 (version 1.20.0)


### 5b. Dense embeddings

In [15]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL, device="cpu")
DENSE_DIM = (getattr(embedder, "get_embedding_dimension", None) or embedder.get_sentence_embedding_dimension)()

t0 = time.time()
dense_vectors = embedder.encode([c.text for c in chunks], batch_size=EMBED_BATCH_SIZE,
                                normalize_embeddings=True, show_progress_bar=True)
print(f"Embedded {len(chunks)} chunks -> {dense_vectors.shape} in {time.time() - t0:.1f}s")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embedded 134 chunks -> (134, 384) in 9.7s


### 5c. Sparse keyword vectors

Term-frequency weights with BM25 saturation; Qdrant multiplies in IDF at query time (`Modifier.IDF`).
Compound tokens such as `LTC3310S-1` are indexed whole **and** split (`ltc3310s`, `1`).
**This function must stay identical in `02_retrieval.ipynb`.**

In [16]:
TOKEN_RE = re.compile(r"[a-z0-9µμ]+(?:[.\-/_][a-z0-9µμ]+)*")
STOPWORDS = set("a an and are as at be by for from has have in is it its of on or that the this to was were will with".split())


def sparse_terms(text):
    terms = []
    for tok in TOKEN_RE.findall(text.lower()):
        terms.append(tok)
        if re.search(r"[.\-/_]", tok):
            terms += [t for t in re.split(r"[.\-/_]", tok) if t]
    return [t for t in terms if t not in STOPWORDS]


def term_index(term):
    return zlib.crc32(term.encode("utf-8")) & 0x7FFFFFFF


def sparse_document_vector(text, k1=1.2):
    weights = {}
    for term, tf in Counter(sparse_terms(text)).items():
        idx = term_index(term)
        weights[idx] = weights.get(idx, 0.0) + tf * (k1 + 1) / (tf + k1)
    return models.SparseVector(indices=list(weights), values=list(weights.values()))


sparse_vectors = [sparse_document_vector(c.text) for c in chunks]
print(f"Sparse vectors built; average {sum(len(v.indices) for v in sparse_vectors) / len(sparse_vectors):.0f} terms per chunk")

Sparse vectors built; average 39 terms per chunk


### 5d. Create the collection (once) and upsert this PDF's chunks

In [17]:
def ensure_collection():
    if client.collection_exists(COLLECTION):
        cfg = client.get_collection(COLLECTION).config.params.vectors
        size = cfg["dense"].size if isinstance(cfg, dict) else cfg.size
        assert size == DENSE_DIM, f"Collection '{COLLECTION}' has dim {size}, model gives {DENSE_DIM}: use a new COLLECTION name"
        print(f"Collection '{COLLECTION}' exists")
        return
    client.create_collection(
        collection_name=COLLECTION,
        vectors_config={"dense": models.VectorParams(size=DENSE_DIM, distance=models.Distance.COSINE)},
        sparse_vectors_config={"sparse": models.SparseVectorParams(modifier=models.Modifier.IDF)},
    )
    if QDRANT_MODE == "server":   # payload indexes make filtered search fast (no-op in local mode)
        for key, schema in [("doc_sha", "keyword"), ("file_name", "keyword"), ("parts", "keyword"),
                            ("type", "keyword"), ("page", "integer")]:
            client.create_payload_index(COLLECTION, field_name=key, field_schema=schema)
    print(f"Created collection '{COLLECTION}'")


ensure_collection()

# Re-ingesting the same PDF replaces its points
doc_filter = models.Filter(must=[models.FieldCondition(key="doc_sha", match=models.MatchValue(value=DOC_SHA))])
old = client.count(COLLECTION, count_filter=doc_filter, exact=True).count
if old:
    client.delete(COLLECTION, points_selector=models.FilterSelector(filter=doc_filter), wait=True)
    print(f"Removed {old} existing points for this PDF")

ingested_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
points = [
    models.PointStruct(
        id=str(uuid.uuid5(uuid.NAMESPACE_URL, c.chunk_id)),
        vector={"dense": dense.tolist(), "sparse": sparse},
        payload={
            **asdict(c),
            "doc_sha": DOC_SHA,
            "file_name": doc_meta["file_name"],
            "source_path": doc_meta["source_path"],
            "title": doc_meta["title"],
            "manufacturer": doc_meta["manufacturer"],
            "revision": doc_meta["revision"],
            "doc_parts": doc_meta["parts"],
            "embedding_model": EMBED_MODEL,
            "ingested_at": ingested_at,
        },
    )
    for c, dense, sparse in zip(chunks, dense_vectors, sparse_vectors)
]

BATCH = 64
for i in range(0, len(points), BATCH):
    client.upsert(COLLECTION, points=points[i:i + BATCH], wait=True)
print(f"Upserted {len(points)} points for {doc_meta['file_name']}")

Created collection 'semiconductor_datasheets'
Upserted 134 points for ltc3310s-3310s-1-3310-3310-1.pdf


### 5e. Verify

In [18]:
n_doc = client.count(COLLECTION, count_filter=doc_filter, exact=True).count
n_all = client.count(COLLECTION, exact=True).count
print(f"Points for this PDF: {n_doc} (expected {len(chunks)}) | total in collection: {n_all}")
assert n_doc == len(chunks)

# Smoke test: the PDF's own title should find chunks of this PDF
probe = embedder.encode("Represent this sentence for searching relevant passages: " + doc_meta["title"],
                        normalize_embeddings=True).tolist()
hits = client.query_points(COLLECTION, query=probe, using="dense", limit=3, with_payload=True).points
for h in hits:
    print(f"{h.score:.3f}  {h.payload['file_name']}  p{h.payload['page']}  {h.payload['text'][:90]!r}")

Points for this PDF: 134 (expected 134) | total in collection: 134
0.694  ltc3310s-3310s-1-3310-3310-1.pdf  p1  '[DESCRIPTION]\nThe LTC®3310S is a very small, low noise, monolithic step-down DC/DC convert'
0.693  ltc3310s-3310s-1-3310-3310-1.pdf  p4  '[ELECTRICAL CHARACTERISTICS]\nTop Switch ON-Resistance - LTC3310S/LTC3310S-1/LTC3310/LTC331'
0.689  ltc3310s-3310s-1-3310-3310-1.pdf  p11  '[BLOCK DIAGRAM]\nNote 1: On the LTC3310S-1 the R and R resistors are internal. A B Note 2: '


In [19]:
# Release the connection. Required in "local" mode: embedded Qdrant allows only one process at a time,
# so notebook 2 cannot open the store while this kernel holds it.
client.close()
print("Done. Ingest another PDF by changing PDF_PATH in step 1 and re-running, or open 02_retrieval.ipynb.")

Done. Ingest another PDF by changing PDF_PATH in step 1 and re-running, or open 02_retrieval.ipynb.
